In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import sys
sys.path.append('/media/dll-1/SSD 4TB/code/REEDNet/')  # Replace with real path
from EEGDataLoader import EEGDataset
from models.REEDNetModel import REEDNetModel
import config
from train_eval import eval, train

model_saving_name = "/media/dll-1/SSD 4TB/EEG Datasets/experiment/nouman_conv_2/model/weights/conv_tt_2_tukl.pt"

epochs = 30
batch_size = 64
hidden_size = 20
output_size = 3  # Binary classification
number_of_eeg_channels = 1
eval_batch = 1
threshold = 0.495
active_branches = [2]

/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_dataset = EEGDataset(config.train_data_csv, number_of_eeg_channels=number_of_eeg_channels)
val_dataset = EEGDataset(config.val_data_csv, number_of_eeg_channels=number_of_eeg_channels)
test_dataset = EEGDataset(config.test_data_csv, number_of_eeg_channels=number_of_eeg_channels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)#, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=eval_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=eval_batch, shuffle=False)

In [ ]:
input = train_dataset.__getitem__(0)
input_x1 = input[1].shape
input_x2 = input[2].shape
input_x3 = input[3].shape
criterion = nn.CrossEntropyLoss()

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

print(input_x1,input_x2,input_x3)
input_channels_list = [input_x1[1], input_x2[1], input_x3[1]]

model = REEDNetModel(input_channels_list=input_channels_list, output_size=3)

main_output, aux_outputs = model([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output.shape) 

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:54: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at ../aten/src/ATen/native/Copy.cpp:305.)
  coefficients_cmor = torch.tensor(coefficients_cmor, dtype=torch.float32)
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


torch.Size([1, 1, 400]) torch.Size([1, 33, 14]) torch.Size([1, 30, 400])
Output shape: torch.Size([1, 3])


ConvParallelEEG1DModel(
  (branches): ModuleList(
    (0): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(in_features=32, out_features=3, bias=True)
    )
    (1): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(33, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(

In [4]:
prev_eval_loss = 0
prev_eval_acc = 0


In [5]:
model = model.to(device)

prev_eval_loss, prev_eval_acc = eval(model, val_loader, loss_fn, 1,False,'', active_branches, device)
print("Loss:",prev_eval_loss, "Accuracy",prev_eval_acc)

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Step 0/43343
Step 100/43343
Step 200/43343
Step 300/43343
Step 400/43343
Step 500/43343
Step 600/43343
Step 700/43343
Step 800/43343
Step 900/43343
Step 1000/43343
Step 1100/43343
Step 1200/43343
Step 1300/43343
Step 1400/43343
Step 1500/43343
Step 1600/43343
Step 1700/43343
Step 1800/43343
Step 1900/43343
Step 2000/43343
Step 2100/43343
Step 2200/43343
Step 2300/43343
Step 2400/43343
Step 2500/43343
Step 2600/43343
Step 2700/43343
Step 2800/43343
Step 2900/43343
Step 3000/43343
Step 3100/43343
Step 3200/43343
Step 3300/43343
Step 3400/43343
Step 3500/43343
Step 3600/43343
Step 3700/43343
Step 3800/43343
Step 3900/43343
Step 4000/43343
Step 4100/43343
Step 4200/43343
Step 4300/43343
Step 4400/43343
Step 4500/43343
Step 4600/43343
Step 4700/43343
Step 4800/43343
Step 4900/43343
Step 5000/43343
Step 5100/43343
Step 5200/43343
Step 5300/43343
Step 5400/43343
Step 5500/43343
Step 5600/43343
Step 5700/43343
Step 5800/43343
Step 5900/43343
Step 6000/43343
Step 6100/43343
Step 6200/43343
Step

/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [6]:
result = {"epoch":[],"train_loss":[],"train_acc":[],"eval_loss":[],"eval_acc":[],"test_acc":[],"test_loss":[]}
saved_epoch = 0

In [7]:
for epoch in range(epochs):  # loop over the dataset multiple times
    train_loss, train_acc = train(model, train_loader,optimizer, loss_fn, active_branches, device)
    print(f"Train Loss after epoch {epoch+2}: {train_loss} Train ACC after epoch {epoch}: {train_acc}")
    eval_loss, eval_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    print(f"VAL Loss after epoch {epoch}: {eval_loss} VAL ACC after epoch {epoch}: {eval_acc}")
    # test_loss, test_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    test_loss = eval_loss
    test_acc = eval_acc
    print(f"Test Loss after epoch {epoch}: {test_loss} Test Acc after epoch {epoch}: {test_acc}")

    scheduler.step(eval_loss)
    
    result['epoch'].append(epoch)
    
    result['train_loss'].append(train_loss)
    result['train_acc'].append(train_acc)
    result['eval_loss'].append(eval_loss)
    result['eval_acc'].append(eval_acc)
    result['test_loss'].append(test_loss)
    result['test_acc'].append(test_acc)

    if prev_eval_acc < eval_acc:
      prev_eval_acc = eval_acc
      saved_epoch = epoch
      torch.save(model.state_dict(), model_saving_name)
      print("WEIGHTS SAVED")

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Step 0/4531
Step 100/4531
Step 200/4531
Step 300/4531
Step 400/4531
Step 500/4531
Step 600/4531
Step 700/4531
Step 800/4531
Step 900/4531
Step 1000/4531
Step 1100/4531
Step 1200/4531
Step 1300/4531
Step 1400/4531
Step 1500/4531
Step 1600/4531
Step 1700/4531
Step 1800/4531
Step 1900/4531
Step 2000/4531
Step 2100/4531
Step 2200/4531
Step 2300/4531
Step 2400/4531
Step 2500/4531
Step 2600/4531
Step 2700/4531
Step 2800/4531
Step 2900/4531
Step 3000/4531
Step 3100/4531
Step 3200/4531
Step 3300/4531
Step 3400/4531
Step 3500/4531
Step 3600/4531
Step 3700/4531
Step 3800/4531
Step 3900/4531
Step 4000/4531
Step 4100/4531
Step 4200/4531
Step 4300/4531
Step 4400/4531
Step 4500/4531
Correct Classified: 187991.0 / 289972
Class-wise correct: Class 0 = 124703.0, Class 1 = 26736.0, Class 2 = 36552.0
Train Loss after epoch 2: 0.8692852172649503 Train ACC after epoch 0: 0.6483074227856482
Step 0/74774


/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Step 100/74774
Step 200/74774
Step 300/74774
Step 400/74774
Step 500/74774
Step 600/74774
Step 700/74774
Step 800/74774
Step 900/74774
Step 1000/74774
Step 1100/74774
Step 1200/74774
Step 1300/74774
Step 1400/74774
Step 1500/74774
Step 1600/74774
Step 1700/74774
Step 1800/74774
Step 1900/74774
Step 2000/74774
Step 2100/74774
Step 2200/74774
Step 2300/74774
Step 2400/74774
Step 2500/74774
Step 2600/74774
Step 2700/74774
Step 2800/74774
Step 2900/74774
Step 3000/74774
Step 3100/74774
Step 3200/74774
Step 3300/74774
Step 3400/74774
Step 3500/74774
Step 3600/74774
Step 3700/74774
Step 3800/74774
Step 3900/74774
Step 4000/74774
Step 4100/74774
Step 4200/74774
Step 4300/74774
Step 4400/74774
Step 4500/74774
Step 4600/74774
Step 4700/74774
Step 4800/74774
Step 4900/74774
Step 5000/74774
Step 5100/74774
Step 5200/74774
Step 5300/74774
Step 5400/74774
Step 5500/74774
Step 5600/74774
Step 5700/74774
Step 5800/74774
Step 5900/74774
Step 6000/74774
Step 6100/74774
Step 6200/74774
Step 6300/74774
S

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


...

Step 100/74774
Step 200/74774
Step 300/74774
Step 400/74774
Step 500/74774
Step 600/74774
Step 700/74774
Step 800/74774
Step 900/74774
Step 1000/74774
Step 1100/74774
Step 1200/74774
Step 1300/74774
Step 1400/74774
Step 1500/74774
Step 1600/74774
Step 1700/74774
Step 1800/74774
Step 1900/74774
Step 2000/74774
Step 2100/74774
Step 2200/74774
Step 2300/74774
Step 2400/74774
Step 2500/74774
Step 2600/74774
Step 2700/74774
Step 2800/74774
Step 2900/74774
Step 3000/74774
Step 3100/74774
Step 3200/74774
Step 3300/74774
Step 3400/74774
Step 3500/74774
Step 3600/74774
Step 3700/74774
Step 3800/74774
Step 3900/74774
Step 4000/74774
Step 4100/74774
Step 4200/74774
Step 4300/74774
Step 4400/74774
Step 4500/74774
Step 4600/74774
Step 4700/74774
Step 4800/74774
Step 4900/74774
Step 5000/74774
Step 5100/74774
Step 5200/74774
Step 5300/74774
Step 5400/74774
Step 5500/74774
Step 5600/74774
Step 5700/74774
Step 5800/74774
Step 5900/74774
Step 6000/74774
Step 6100/74774
Step 6200/74774
Step 6300/74774
S

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Step 100/4531
Step 200/4531
Step 300/4531
Step 400/4531
Step 500/4531
Step 600/4531
Step 700/4531
Step 800/4531
Step 900/4531
Step 1000/4531
Step 1100/4531
Step 1200/4531
Step 1300/4531
Step 1400/4531
Step 1500/4531
Step 1600/4531
Step 1700/4531
Step 1800/4531
Step 1900/4531
Step 2000/4531
Step 2100/4531
Step 2200/4531
Step 2300/4531
Step 2400/4531
Step 2500/4531
Step 2600/4531
Step 2700/4531
Step 2800/4531
Step 2900/4531
Step 3000/4531
Step 3100/4531
Step 3200/4531
Step 3300/4531
Step 3400/4531
Step 3500/4531
Step 3600/4531
Step 3700/4531
Step 3800/4531
Step 3900/4531
Step 4000/4531
Step 4100/4531
Step 4200/4531
Step 4300/4531
Step 4400/4531
Step 4500/4531
Correct Classified: 222038.0 / 289972
Class-wise correct: Class 0 = 119467.0, Class 1 = 40583.0, Class 2 = 61988.0
Train Loss after epoch 31: 0.7090336485325803 Train ACC after epoch 29: 0.7657222076614294
Step 0/74774


/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Step 100/74774
Step 200/74774
Step 300/74774
Step 400/74774
Step 500/74774
Step 600/74774
Step 700/74774
Step 800/74774
Step 900/74774
Step 1000/74774
Step 1100/74774
Step 1200/74774
Step 1300/74774
Step 1400/74774
Step 1500/74774
Step 1600/74774
Step 1700/74774
Step 1800/74774
Step 1900/74774
Step 2000/74774
Step 2100/74774
Step 2200/74774
Step 2300/74774
Step 2400/74774
Step 2500/74774
Step 2600/74774
Step 2700/74774
Step 2800/74774
Step 2900/74774
Step 3000/74774
Step 3100/74774
Step 3200/74774
Step 3300/74774
Step 3400/74774
Step 3500/74774
Step 3600/74774
Step 3700/74774
Step 3800/74774
Step 3900/74774
Step 4000/74774
Step 4100/74774
Step 4200/74774
Step 4300/74774
Step 4400/74774
Step 4500/74774
Step 4600/74774
Step 4700/74774
Step 4800/74774
Step 4900/74774
Step 5000/74774
Step 5100/74774
Step 5200/74774
Step 5300/74774
Step 5400/74774
Step 5500/74774
Step 5600/74774
Step 5700/74774
Step 5800/74774
Step 5900/74774
Step 6000/74774
Step 6100/74774
Step 6200/74774
Step 6300/74774
S

In [8]:
print("Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  ")
for epoch in range(epochs):
    if(saved_epoch == epoch):
        print("Saved Weights")
    print(result['epoch'][epoch],"|",result['train_loss'][epoch],"|",result['train_acc'][epoch],"|",result['eval_loss'][epoch],"|",result['eval_acc'][epoch],"|",result['test_loss'][epoch],"|",result['test_acc'][epoch])

Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  
0 | 0.8692852172649503 | 0.6483074227856482 | 0.8303890721806935 | 0.6186909888463905 | 0.8303890721806935 | 0.6186909888463905
1 | 0.7756448754067716 | 0.7162208764984206 | 0.7925819525919694 | 0.6554417310829967 | 0.7925819525919694 | 0.6554417310829967
2 | 0.7552755723670996 | 0.7356020581297504 | 0.7758876090135898 | 0.694024660978415 | 0.7758876090135898 | 0.694024660978415
3 | 0.7431985346114917 | 0.7446891423999559 | 0.7732303249470722 | 0.6974081900125713 | 0.7732303249470722 | 0.6974081900125713
4 | 0.7361774774635372 | 0.7482619011490764 | 0.7577492911227616 | 0.6936903201647632 | 0.7577492911227616 | 0.6936903201647632
Saved Weights
5 | 0.731648742376482 | 0.7516311919771564 | 0.7524929472593548 | 0.7102201299917084 | 0.7524929472593548 | 0.7102201299917084
6 | 0.7286676530594911 | 0.753841750237954 | 0.7549294673409986 | 0.7067831064273677 | 0.7549294673409986 | 0.7067831064273677
7 | 0.725615131